# 04 — Exportación Anonimizada

Lee `out/caracterizaciones_limpio.db` y `out/comunidades_limpio.db` (Fase 3).
Si no existen, cae back a los DBs de Fase 1 (`caracterizaciones.db` / `comunidades.db`).

Produce `out/comunidad_anonimizada.db` con campos PII hasheados por token.

**Algoritmo**: cada campo PII se tokeniza por espacios. Cada token se normaliza
(lowercase + sin acentos) y se reemplaza por los primeros 8 caracteres de su SHA1.
Los campos NO son eliminados — solo seudonimizados.

Ver spec: `specs/20260414-anonimizacion-token-hash.md`

In [1]:
# --- Parámetros (editables via nbclick o papermill) ---

# Inputs: usa versión limpia (Fase 3) si existe, si no cae a Fase 1
INPUT_CARACTERIZACIONES_LIMPIO = "out/caracterizaciones_limpio.db"
INPUT_CARACTERIZACIONES_RAW    = "out/caracterizaciones.db"
INPUT_COMUNIDADES_LIMPIO       = "out/comunidades_limpio.db"
INPUT_COMUNIDADES_RAW          = "out/comunidades.db"

OUTPUT_ANONIMIZADA = "out/comunidad_anonimizada.db"

# Columnas PII a anonimizar — nombres sanitizados (snake_case, sin acentos)
# tal como los escribe write_to_sqlite en el notebook 01.
PII_CARACTERIZACIONES = [
    "nombre_del_interesado",
    "documento_de_identificacion_del_interesado",
    "celular_del_interesado",
    "celular_de_otro_contacto_del_interesado",
    "nombre_de_otro_contacto_del_interesado",
    "nombre_del_voluntario",
    "celular_voluntario",
]

PII_COMUNIDADES = [
    "cedula",
    "nombre_completo",
    "nombre_contacto_2",
    "contacto_1",
    "contacto_2",
    "detalle_direccion",
]

In [2]:
import hashlib
import logging
import sqlite3
import sys
import unicodedata
from pathlib import Path

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[logging.StreamHandler(sys.stdout)],
)
log = logging.getLogger("anonimizacion")


def find_project_root(marker="pyproject.toml"):
    for parent in [Path.cwd(), *Path.cwd().parents]:
        if (parent / marker).exists():
            return parent
    raise RuntimeError(f"No se encontró la raíz del proyecto (buscando '{marker}' desde {Path.cwd()})")


PROJECT_ROOT = find_project_root()
log.info(f"Project root: {PROJECT_ROOT}")


def resolve_input(preferred: str, fallback: str) -> Path:
    """Usa la versión limpia (Fase 3) si existe; si no, la de Fase 1."""
    p = PROJECT_ROOT / preferred
    if p.exists():
        log.info(f"Input: {p.relative_to(PROJECT_ROOT)} (versión limpia Fase 3)")
        return p
    fb = PROJECT_ROOT / fallback
    if fb.exists():
        log.warning(f"Fase 3 no encontrada — usando Fase 1: {fb.relative_to(PROJECT_ROOT)}")
        return fb
    raise FileNotFoundError(
        f"No se encontró ni '{preferred}' ni '{fallback}'. Ejecutar Fase 1 primero."
    )


in_car  = resolve_input(INPUT_CARACTERIZACIONES_LIMPIO, INPUT_CARACTERIZACIONES_RAW)
in_com  = resolve_input(INPUT_COMUNIDADES_LIMPIO,       INPUT_COMUNIDADES_RAW)
out_anon = PROJECT_ROOT / OUTPUT_ANONIMIZADA

log.info(f"Output: {out_anon.relative_to(PROJECT_ROOT)}")

2026-04-14 10:14:56,813 [INFO] Project root: /home/aldrin/projetos/sources/techo-cleanup


2026-04-14 10:14:56,814 [WARNING] Fase 3 no encontrada — usando Fase 1: out/caracterizaciones.db


2026-04-14 10:14:56,815 [WARNING] Fase 3 no encontrada — usando Fase 1: out/comunidades.db


2026-04-14 10:14:56,815 [INFO] Output: out/comunidad_anonimizada.db


In [3]:
# --- Funciones de anonimización ---

def anon_token(s: str) -> str:
    """
    Convierte un token en su hash seudonimizado de 8 caracteres hex.

    Pasos:
    1. Normalizar unicode NFD y eliminar marcas de acento (categoria Mn)
    2. Lowercase
    3. SHA1 → primeros 8 caracteres del hexdigest

    Ejemplos:
      anon_token("Juan")  == anon_token("juan")   # case-insensitive
      anon_token("Pérez") == anon_token("Perez")  # acento-insensitive
    """
    normalized = unicodedata.normalize("NFD", s)
    without_accents = "".join(c for c in normalized if unicodedata.category(c) != "Mn")
    lower = without_accents.lower()
    return hashlib.sha1(lower.encode("utf-8")).hexdigest()[:8]


def anon_field(value) -> str | None:
    """
    Tokeniza un campo por espacios y hashea cada token.
    Preserva None y strings vacíos sin modificación.

    Ejemplo: "Juan Pérez" → "<hash_juan> <hash_perez>"
    """
    if value is None:
        return None
    s = str(value).strip()
    if not s:
        return value
    return " ".join(anon_token(t) for t in s.split())


# --- Tests inline ---
assert anon_token("Juan") == anon_token("juan"),   "case-insensitive falla"
assert anon_token("Pérez") == anon_token("Perez"), "normalización de acentos falla"
assert anon_token("pérez") == anon_token("perez"), "acento + lower falla"
assert len(anon_token("x")) == 8,                  "longitud del hash debe ser 8"
assert anon_field(None) is None,                   "None debe preservarse"
assert anon_field("") == "",                        "string vacío debe preservarse"
parts = anon_field("Juan Pérez").split()
assert len(parts) == 2,                             "multi-token debe producir 2 hashes"
assert all(len(p) == 8 for p in parts),            "cada hash debe tener 8 chars"

log.info("✓ Tests inline de anon_token / anon_field pasaron")
log.info(f"  Ejemplo: 'Juan Pérez' → '{anon_field('Juan Pérez')}'")
log.info(f"  Ejemplo: '12345678'   → '{anon_field('12345678')}'")

2026-04-14 10:14:56,821 [INFO] ✓ Tests inline de anon_token / anon_field pasaron


2026-04-14 10:14:56,822 [INFO]   Ejemplo: 'Juan Pérez' → 'b49a5780 39caf24e'


2026-04-14 10:14:56,822 [INFO]   Ejemplo: '12345678'   → '7c222fb2'


In [4]:
# --- Leer tabla SQLite y detectar columnas PII presentes ---

def read_table(db_path: Path, table: str) -> tuple[list[str], list[dict]]:
    """Lee todas las filas de una tabla SQLite. Retorna (columnas, filas)."""
    with sqlite3.connect(db_path) as conn:
        conn.row_factory = sqlite3.Row
        cur = conn.execute(f'SELECT * FROM "{table}"')
        rows = [dict(r) for r in cur.fetchall()]
        cols = [d[0] for d in cur.description]
    return cols, rows


def get_tables(db_path: Path) -> list[str]:
    with sqlite3.connect(db_path) as conn:
        cur = conn.execute("SELECT name FROM sqlite_master WHERE type='table'")
        return [r[0] for r in cur.fetchall()]


def write_table(db_path: Path, table: str, cols: list[str], rows: list[dict]):
    """Escribe tabla en SQLite (idempotente — DROP IF EXISTS + CREATE)."""
    db_path.parent.mkdir(parents=True, exist_ok=True)
    with sqlite3.connect(db_path) as conn:
        conn.execute(f'DROP TABLE IF EXISTS "{table}"')
        col_defs = ", ".join(f'"{c}" TEXT' for c in cols)
        conn.execute(f'CREATE TABLE "{table}" ({col_defs})')
        placeholders = ", ".join(["?"] * len(cols))
        for row in rows:
            vals = [row.get(c) for c in cols]
            conn.execute(f'INSERT INTO "{table}" VALUES ({placeholders})', vals)
        conn.commit()


def anonimizar_tabla(
    db_path: Path,
    table: str,
    pii_cols: list[str],
    label: str,
) -> tuple[list[str], list[dict], dict]:
    """
    Lee la tabla, aplica anon_field a las columnas PII presentes.
    Retorna (cols, rows_anonimizados, stats).
    """
    cols, rows = read_table(db_path, table)
    log.info(f"{label}: {len(rows)} filas, {len(cols)} columnas")

    # Detectar qué columnas PII realmente existen en la tabla
    pii_present = [c for c in pii_cols if c in cols]
    pii_missing  = [c for c in pii_cols if c not in cols]

    log.info(f"  PII encontradas ({len(pii_present)}): {pii_present}")
    if pii_missing:
        log.info(f"  PII no presentes en tabla (se ignoran): {pii_missing}")

    stats = {"filas": len(rows), "cols_anonimizadas": pii_present, "campos_modificados": 0}

    for row in rows:
        for col in pii_present:
            original = row.get(col)
            anonimizado = anon_field(original)
            if anonimizado != original:
                stats["campos_modificados"] += 1
            row[col] = anonimizado

    log.info(f"  Campos modificados: {stats['campos_modificados']} de {len(rows) * len(pii_present)} posibles")
    return cols, rows, stats


log.info("Funciones de anonimización cargadas.")

2026-04-14 10:14:56,831 [INFO] Funciones de anonimización cargadas.


In [5]:
# --- Anonimizar caracterizaciones ---
log.info("\n=== Anonimizando: caracterizaciones ===")

tables_car = get_tables(in_car)
table_car  = tables_car[0]  # esperamos 'caracterizaciones'
log.info(f"Tabla: '{table_car}'")

car_cols, car_rows, car_stats = anonimizar_tabla(
    in_car, table_car, PII_CARACTERIZACIONES, "caracterizaciones"
)

2026-04-14 10:14:56,835 [INFO] 
=== Anonimizando: caracterizaciones ===


2026-04-14 10:14:56,836 [INFO] Tabla: 'caracterizaciones'


2026-04-14 10:14:56,843 [INFO] caracterizaciones: 558 filas, 29 columnas


2026-04-14 10:14:56,843 [INFO]   PII encontradas (7): ['nombre_del_interesado', 'documento_de_identificacion_del_interesado', 'celular_del_interesado', 'celular_de_otro_contacto_del_interesado', 'nombre_de_otro_contacto_del_interesado', 'nombre_del_voluntario', 'celular_voluntario']


2026-04-14 10:14:56,855 [INFO]   Campos modificados: 3177 de 3906 posibles


In [6]:
# --- Anonimizar comunidades ---
log.info("\n=== Anonimizando: comunidades ===")

tables_com = get_tables(in_com)
table_com  = tables_com[0]  # esperamos 'comunidades'
log.info(f"Tabla: '{table_com}'")

com_cols, com_rows, com_stats = anonimizar_tabla(
    in_com, table_com, PII_COMUNIDADES, "comunidades"
)

2026-04-14 10:14:56,860 [INFO] 
=== Anonimizando: comunidades ===


2026-04-14 10:14:56,860 [INFO] Tabla: 'comunidades'


2026-04-14 10:14:56,881 [INFO] comunidades: 744 filas, 43 columnas


2026-04-14 10:14:56,881 [INFO]   PII encontradas (6): ['cedula', 'nombre_completo', 'nombre_contacto_2', 'contacto_1', 'contacto_2', 'detalle_direccion']


2026-04-14 10:14:56,895 [INFO]   Campos modificados: 3075 de 4464 posibles


In [7]:
# --- Escribir output anonimizado ---
log.info("\n=== Escribiendo output ===")

write_table(out_anon, "caracterizaciones", car_cols, car_rows)
log.info(f"✓ caracterizaciones: {len(car_rows)} filas")

write_table(out_anon, "comunidades", com_cols, com_rows)
log.info(f"✓ comunidades:       {len(com_rows)} filas")

2026-04-14 10:14:56,899 [INFO] 
=== Escribiendo output ===


2026-04-14 10:14:56,951 [INFO] ✓ caracterizaciones: 558 filas


2026-04-14 10:14:56,975 [INFO] ✓ comunidades:       744 filas


In [8]:
# --- Verificación: JOIN por campo cédula hasheado ---
log.info("\n=== Verificación de JOIN por cédula hasheada ===")

# Buscar columna de cédula en cada tabla (nombre sanitizado)
CEDULA_CAR = next((c for c in car_cols if "documento" in c or "cedula" in c), None)
CEDULA_COM = next((c for c in com_cols if "cedula" in c), None)

if CEDULA_CAR and CEDULA_COM:
    ids_car = {r.get(CEDULA_CAR) for r in car_rows if r.get(CEDULA_CAR)}
    ids_com = {r.get(CEDULA_COM) for r in com_rows if r.get(CEDULA_COM)}
    matched = ids_car & ids_com
    log.info(f"  Cédula hash en caracterizaciones: {len(ids_car)} únicos")
    log.info(f"  Cédula hash en comunidades:       {len(ids_com)} únicos")
    log.info(f"  Match cross-tabla (JOIN posible):  {len(matched)}")
    if len(matched) == 0:
        log.warning("  ⚠ Sin matches — revisar nombres de columnas PII_CARACTERIZACIONES / PII_COMUNIDADES")
else:
    log.warning(f"  ⚠ Columna cédula no encontrada — car={CEDULA_CAR}, com={CEDULA_COM}")
    log.warning(f"  Columnas car: {car_cols}")
    log.warning(f"  Columnas com: {com_cols}")

2026-04-14 10:14:56,981 [INFO] 
=== Verificación de JOIN por cédula hasheada ===


2026-04-14 10:14:56,982 [INFO]   Cédula hash en caracterizaciones: 301 únicos


2026-04-14 10:14:56,982 [INFO]   Cédula hash en comunidades:       504 únicos


2026-04-14 10:14:56,983 [INFO]   Match cross-tabla (JOIN posible):  273


In [9]:
# --- Resumen final ---
log.info("\n=== RESUMEN DE ANONIMIZACIÓN ===")
log.info(f"Output: {out_anon.relative_to(PROJECT_ROOT)}")
log.info(f"caracterizaciones: {car_stats['filas']} filas, "
         f"cols PII: {car_stats['cols_anonimizadas']}, "
         f"campos modificados: {car_stats['campos_modificados']}")
log.info(f"comunidades:       {com_stats['filas']} filas, "
         f"cols PII: {com_stats['cols_anonimizadas']}, "
         f"campos modificados: {com_stats['campos_modificados']}")
log.info("Anonimización completada. ✓")

2026-04-14 10:14:56,987 [INFO] 
=== RESUMEN DE ANONIMIZACIÓN ===


2026-04-14 10:14:56,988 [INFO] Output: out/comunidad_anonimizada.db


2026-04-14 10:14:56,988 [INFO] caracterizaciones: 558 filas, cols PII: ['nombre_del_interesado', 'documento_de_identificacion_del_interesado', 'celular_del_interesado', 'celular_de_otro_contacto_del_interesado', 'nombre_de_otro_contacto_del_interesado', 'nombre_del_voluntario', 'celular_voluntario'], campos modificados: 3177


2026-04-14 10:14:56,988 [INFO] comunidades:       744 filas, cols PII: ['cedula', 'nombre_completo', 'nombre_contacto_2', 'contacto_1', 'contacto_2', 'detalle_direccion'], campos modificados: 3075


2026-04-14 10:14:56,988 [INFO] Anonimización completada. ✓
